In [0]:
# =============================================================================
# CELL 1 — IMPORTS & SPARK CONFIGURATION
# =============================================================================
 
import logging
import json
 
from datetime import datetime, timezone
from typing   import Optional
 
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType
 
logging.basicConfig(
    level  = logging.INFO,
    format = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt= "%Y-%m-%dT%H:%M:%SZ"
)
logger = logging.getLogger("bronze_autoloader")
 
# Enable schema evolution: if CoinGecko adds a new top-level field to their
# response in the future, Autoloader will add a new column to the Delta table
# instead of failing with a schema mismatch error.
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
 
logger.info("Spark configured: schema auto-merge ON")

In [0]:
# =============================================================================
# CELL 2 — CONFIGURATION  (all paths, no catalog names)
# =============================================================================
 
ADLS_NAME = "adlsnewhp1"
 
# ── ADLS container roots ──────────────────────────────────────────────────────
LANDING_ROOT = f"abfss://capstone@{ADLS_NAME}.dfs.core.windows.net"
BRONZE_ROOT  = f"abfss://capstone@{ADLS_NAME}.dfs.core.windows.net/bronze"
 
# ── Checkpoint root ───────────────────────────────────────────────────────────
# Checkpoints MUST live outside the Delta table folders.
# If they lived inside, a Delta VACUUM run could delete them and break
# Autoloader's incremental file tracking permanently.
# We put them in bronze/_autoloader_checkpoints/ — a dedicated sub-path
# that Delta operations never touch.
CHECKPOINT_ROOT = f"{BRONZE_ROOT}/_autoloader_checkpoints"
 
# ── Schema inference cache ────────────────────────────────────────────────────
# Autoloader infers schema from the first batch of files and caches it here.
# On subsequent runs it reads the cached schema (fast) rather than
# re-inferring from scratch (slow + potentially inconsistent across runs).
SCHEMA_ROOT = f"{BRONZE_ROOT}/_autoloader_schemas"
 
NOW_UTC = datetime.now(timezone.utc)
 
logger.info(f"Landing Root    : {LANDING_ROOT}")
logger.info(f"Bronze Root     : {BRONZE_ROOT}")
logger.info(f"Checkpoint Root : {CHECKPOINT_ROOT}")
logger.info(f"Schema Root     : {SCHEMA_ROOT}")
logger.info(f"Run Timestamp   : {NOW_UTC.isoformat()}")
 
